In [1]:
import numpy as np
import torch
import os
from pathlib import Path
import matplotlib.pyplot as plt
# from data_preparation.field import load_fields

In [2]:
processed_data_dir = Path("../processed_data")
exp_name = "history1_fv"
case_name = "flange"
case_name = f"{case_name}_{exp_name}"
case_dir = processed_data_dir / case_name
checkpoint_dir = case_dir / "checkpoints"
pred_dir = case_dir / "predictions"
error_dir = case_dir / "errors"

In [3]:
train_loss = np.load(os.path.join(checkpoint_dir, "train_losses.npy"))
val_loss = np.load(os.path.join(checkpoint_dir, "val_losses.npy"))
rollout_mae = np.load(os.path.join(checkpoint_dir, "rollout_mae.npy"))

In [4]:
AX_COLOR = "#1e2a2c"
LEGEND_BG = "#f0eee9"
PLOT1_COLOR = "#5e3d80"
PLOT2_COLOR = "#174535"

px = 1/plt.rcParams['figure.dpi']  # pixel in inches

loss_graph, ax = plt.subplots(figsize=(960*px, 540*px))

# Transparent background
loss_graph.patch.set_alpha(0)
ax.patch.set_alpha(0)

ax.title.set_color(AX_COLOR)
ax.xaxis.label.set_color(AX_COLOR)
ax.yaxis.label.set_color(AX_COLOR)
ax.tick_params(colors=AX_COLOR)
for spine in ax.spines.values():
    spine.set_edgecolor(AX_COLOR)

plt.plot(train_loss, label="Train Loss", color = PLOT1_COLOR, linewidth=2.5)
plt.plot(val_loss, label="Validation Loss", color = PLOT2_COLOR, linewidth=2.5)
ax.set(yscale='log')

legend = ax.legend(fontsize=18)
legend.get_frame().set_facecolor(LEGEND_BG)   # background fill
legend.get_frame().set_edgecolor(AX_COLOR)        # border color
legend.get_frame().set_alpha(1)                # fully opaque (or 0–1)
for text in legend.get_texts():
    text.set_color(AX_COLOR)                      # label text color

plt.xlabel("Epoch")
plt.ylabel("Loss")
ax.set_title("Training and Validation Loss, h = 1, FV", fontsize=24)
ax.set_yscale("log")
ax.set_xlabel("Epoch", fontsize=18)
ax.set_ylabel("MSE Loss", fontsize=18)
ax.tick_params(axis='both', labelsize=16)
ax.grid(True, which="both", ls="--", linewidth=0.5, color=AX_COLOR, alpha=0.7)


# plt.savefig(f"../outputs/plots/{case_name}.png", dpi=300, transparent = True)

In [5]:
train_loss, val_loss, rollout_mae = {},{},{}
processed_data_dir = Path("../processed_data")
for case_name in ["flange_history1_fv", "flange_history1_nofv"]:
    case_dir = processed_data_dir / case_name
    checkpoint_dir = case_dir / "checkpoints"
    pred_dir = case_dir / "predictions"
    error_dir = case_dir / "errors"

    train_loss[case_name] = np.load(os.path.join(checkpoint_dir, "train_losses.npy"))
    val_loss[case_name] = np.load(os.path.join(checkpoint_dir, "val_losses.npy"))
    rollout_mae[case_name] = np.load(os.path.join(checkpoint_dir, "rollout_mae.npy"))

In [6]:
fig, axs = plt.subplots(1,1)
axs.set_title("Training Loss", fontsize=20)
axs.set_yscale("log")
axs.set_xlabel("Epoch", fontsize=18)
axs.set_ylabel("MSE Loss", fontsize=18)
for i,case_name in enumerate(train_loss):
    plt.plot(train_loss[case_name], label=f"{case_name}: Train Loss")
    plt.plot(val_loss[case_name], label=f"{case_name}: Validation Loss")

axs.legend(fontsize=14)

In [7]:
AX_COLOR = "#1e2a2c"
LEGEND_BG = "#f0eee9"
PLOT1_COLOR = "#5e3d80"
PLOT2_COLOR = "#174535"

px = 1/plt.rcParams['figure.dpi']  # pixel in inches

fig, axs = plt.subplots(1,1, figsize=(960*px, 540*px))


# Transparent background
loss_graph.patch.set_alpha(0)
axs.patch.set_alpha(0)

axs.title.set_color(AX_COLOR)
axs.xaxis.label.set_color(AX_COLOR)
axs.yaxis.label.set_color(AX_COLOR)
axs.tick_params(colors=AX_COLOR)
for spine in axs.spines.values():
    spine.set_edgecolor(AX_COLOR)


for i,case_name in enumerate(train_loss):
    if case_name.endswith("_fv"):
        plt.plot(rollout_mae[case_name], label="FV", color = PLOT1_COLOR, linewidth=2.5)
    elif case_name.endswith("_nofv"):
        plt.plot(rollout_mae[case_name], label="No FV", color = PLOT2_COLOR, linewidth=2.5, linestyle='--')

legend = axs.legend(fontsize=18)
legend.get_frame().set_facecolor(LEGEND_BG)   # background fill
legend.get_frame().set_edgecolor(AX_COLOR)        # border color
legend.get_frame().set_alpha(1)                # fully opaque (or 0–1)
for text in legend.get_texts():
    text.set_color(AX_COLOR)                      # label text color

axs.set_title("Rollout Error Accumulation, h = 1", fontsize=24)
axs.set_xlabel("Time step", fontsize=18)
axs.set_ylabel("MAE", fontsize=18)
axs.tick_params(axis='both', labelsize=16)
axs.grid(True, which="both", ls="--", linewidth=0.5, color=AX_COLOR, alpha=0.7)

# plt.savefig(f"../outputs/plots/{case_name[:15]}_rollout.png", dpi=300, transparent = True)


## Parametric case

Training/validation-loss and rollout-error plots for the parametric runs at `history = 1`.
The rollout panels mirror the flange **FV vs no-FV** comparison
(`parametric_history1_mesh_correct_edge_attr` vs `parametric_history1_mesh_nofv`),
one panel per mesh held out from those checkpoints: `model_003` and `model_007`
(the original 10-mesh study's test set) plus `model_010` and `model_011`
(the 2-hole / 3-hole geometries added later, which postdate both checkpoints).

In [8]:
# --- Parametric case (history = 1, FV vs no-FV) ------------------------
# Each parametric run is tested on the held-out meshes model_003 and
# model_007, so there is one rollout-MAE curve per test mesh.
param_cases = {
    "fv":   "parametric_history1_mesh_correct_edge_attr",
    "nofv": "parametric_history1_mesh_nofv",
}
param_test_meshes = ["model_003", "model_007", "model_010", "model_011"]

# The FV run drives the single-case loss plot below.
param_case = param_cases["fv"]
param_ckpt = processed_data_dir / param_case / "checkpoints"
param_train_loss = np.load(os.path.join(param_ckpt, "train_losses.npy"))
param_val_loss = np.load(os.path.join(param_ckpt, "val_losses.npy"))

# rollout MAE per {fv/nofv} run and per test mesh.
param_rollout = {}
for key, cname in param_cases.items():
    ckpt = processed_data_dir / cname / "checkpoints"
    param_rollout[key] = {
        mesh: np.load(os.path.join(ckpt, f"rollout_mae_{mesh}.npy"))
        for mesh in param_test_meshes
    }


In [9]:
AX_COLOR = "#1e2a2c"
LEGEND_BG = "#f0eee9"
PLOT1_COLOR = "#5e3d80"
PLOT2_COLOR = "#174535"

px = 1/plt.rcParams['figure.dpi']  # pixel in inches

loss_graph, ax = plt.subplots(figsize=(960*px, 540*px))

# Transparent background
loss_graph.patch.set_alpha(0)
ax.patch.set_alpha(0)

ax.title.set_color(AX_COLOR)
ax.xaxis.label.set_color(AX_COLOR)
ax.yaxis.label.set_color(AX_COLOR)
ax.tick_params(colors=AX_COLOR)
for spine in ax.spines.values():
    spine.set_edgecolor(AX_COLOR)

plt.plot(param_train_loss, label="Train Loss", color = PLOT1_COLOR, linewidth=2.5)
plt.plot(param_val_loss, label="Validation Loss", color = PLOT2_COLOR, linewidth=2.5)
ax.set(yscale='log')

legend = ax.legend(fontsize=18)
legend.get_frame().set_facecolor(LEGEND_BG)   # background fill
legend.get_frame().set_edgecolor(AX_COLOR)        # border color
legend.get_frame().set_alpha(1)                # fully opaque (or 0-1)
for text in legend.get_texts():
    text.set_color(AX_COLOR)                      # label text color

plt.xlabel("Epoch")
plt.ylabel("Loss")
ax.set_title("Training and Validation Loss, Parametric FV, h = 1", fontsize=24)
ax.set_yscale("log")
ax.set_xlabel("Epoch", fontsize=18)
ax.set_ylabel("MSE Loss", fontsize=18)
ax.tick_params(axis='both', labelsize=16)
ax.grid(True, which="both", ls="--", linewidth=0.5, color=AX_COLOR, alpha=0.7)


# plt.savefig(f"../outputs/plots/{param_case}.png", dpi=300, transparent = True)


In [10]:
from matplotlib.ticker import StrMethodFormatter

AX_COLOR = "#1e2a2c"
LEGEND_BG = "#f0eee9"
PLOT1_COLOR = "#5e3d80"
PLOT2_COLOR = "#174535"

px = 1/plt.rcParams['figure.dpi']  # pixel in inches

# One panel per held-out test mesh; each panel is the flange-style
# FV vs no-FV rollout comparison.
fig, axs = plt.subplots(len(param_test_meshes),1,
                        figsize=(960*px, len(param_test_meshes)*540*px),
                        constrained_layout=True)

# Transparent background
fig.patch.set_alpha(0)

for ax, mesh in zip(axs, param_test_meshes[::-1]):
    ax.patch.set_alpha(0)
    ax.title.set_color(AX_COLOR)
    ax.xaxis.label.set_color(AX_COLOR)
    ax.yaxis.label.set_color(AX_COLOR)
    ax.tick_params(colors=AX_COLOR)
    for spine in ax.spines.values():
        spine.set_edgecolor(AX_COLOR)

    fv_curve = param_rollout["fv"][mesh]
    nofv_curve = param_rollout["nofv"][mesh]
    ax.plot(fv_curve, label="FV",
            color=PLOT1_COLOR, linewidth=2.5)
    ax.plot(nofv_curve, label="No FV",
            color=PLOT2_COLOR, linewidth=2.5, linestyle='--')

    # Log-scale whenever the two curves span more than ~1.5 decades, otherwise
    # the FV curve is squashed onto the axis by the diverging no-FV one. Was
    # hardcoded to model_003; made data-driven so added meshes are handled too.
    both = np.concatenate([fv_curve, nofv_curve])
    if both.max() / max(both.min(), 1e-30) > 30:
        ax.set_yscale("log")
        ax.yaxis.set_major_formatter(StrMethodFormatter('{x:.1e}'))

    legend = ax.legend(fontsize=18)
    legend.get_frame().set_facecolor(LEGEND_BG)   # background fill
    legend.get_frame().set_edgecolor(AX_COLOR)        # border color
    legend.get_frame().set_alpha(1)                # fully opaque (or 0-1)
    for text in legend.get_texts():
        text.set_color(AX_COLOR)                      # label text color

    ax.set_title(f"Rollout Error Accumulation, {mesh}, h = 1", fontsize=24)
    ax.set_xlabel("Time step", fontsize=18)
    ax.set_ylabel("MAE", fontsize=18)
    ax.tick_params(axis='both', labelsize=16)
    ax.grid(True, which="both", ls="--", linewidth=0.5, color=AX_COLOR, alpha=0.7)

# plt.tight_layout(pad=0)
# plt.savefig("../outputs/plots/parametric_history1_mesh_rollout_exp.png", dpi=300, transparent = True)

## Message vs. finite-volume flux

`FiniteVolumeGraphNet` sums its messages at the receiving node the way a finite-volume
scheme sums face fluxes over a cell, so the obvious question is whether the learned
$m_{ij}$ *is* the flux `laplacianFoam` assembles for `laplacian(DT,T)` with
`Gauss linear corrected`:

$$F_{j\rightarrow i} \;=\; D_T\,|S_f|\left[\;\frac{T_j - T_i}{|d|\,\max(\cos,\,0.05)} \;+\; k_f\cdot(\nabla T)_f\;\right]$$

Everything below runs on the **parametric** study — the same two `history = 1`
checkpoints the rollout panels compare (`parametric_history1_mesh_correct_edge_attr`
vs `parametric_history1_mesh_nofv`), probed on the same four held-out meshes:
`model_003` and `model_007` from the recorded split, plus the later `model_010` /
`model_011`. Nothing in the split is temporal, so every timestep of those meshes is
test data and the probe is free to sample the whole sequence.

`propagate` throws $m_{ij}$ away before `forward` returns, so it is pulled back out with
`models.message_probe.MessageProbe`, which hangs a PyG `register_message_forward_hook` on
each MP layer. Nothing in `fvgnn.py` or the `state_dict` changes, so every checkpoint
already on disk is probed as-is:

```python
probe = MessageProbe(model)
with probe, torch.no_grad():
    for i in range(n): model(data_i)

probe.messages[0]            # (E, 128)    last pass, layer 0
probe.stacked_messages(0)    # (n, E, 128) whole rollout
probe.stacked_aggregated(0)  # (n, N, 128) sum_j m_ij
```

`analyze_messages.py` runs the same comparison from the command line.

### Orientation — the thing that silently breaks this

PyG aggregates at `edge_index[1]`, so inside `message()` **`x_i` is the receiver and
`x_j` the sender**, and `build_static_graph` already orients each edge's stored $S_f$
*into* the receiver. `fv_flux` follows the same convention: a positive flux heats the
receiver, and summing over the in-edges of node $i$ gives $V_i\,\mathrm{d}T_i/\mathrm{d}t$.
Get this backwards and every correlation below flips sign while still looking plausible.

In [11]:
import sys
print(sys.executable)
print(torch.__file__)

In [12]:
# --- Message vs FV flux: extract, and cache -------------------------------
# Runs both trained parametric models over N_STEPS timesteps of every held-out
# mesh with a MessageProbe attached, and stores everything the figures below
# need. A parametric mesh carries up to ~270k edges, so a full (steps, E, 128)
# message stack is several GB: only a random subset of whole faces is held in
# memory, and only the reduced statistics reach the cache.

import torch

from data_preparation.field import load_fields
from data_preparation.mesh_dataset import SingleMeshDataset
from data_preparation.normalization import FeatureNormalizer
from data_preparation.static_graph import build_static_graph
from mesh2graph.utils import filter_of_time_directories
from models.fvgnn import FVSurrogate
from models.message_probe import (MessageProbe, fv_flux, fv_flux_corrected,
                                  load_cell_gradient, paired_edge_index)

FLUX_EXPS   = {"FV": "parametric_history1_mesh_correct_edge_attr",
               "No FV": "parametric_history1_mesh_nofv"}
# The same held-out meshes the rollout panels above compare.
FLUX_MESHES = ["model_003", "model_007", "model_010", "model_011"]
EXCLUDED    = ["top", "bottom", "cbores"]
DT_PARAM    = 4e-5          # constant/transportProperties
HISTORY     = 1
# The mesh split is over GEOMETRY, not time, so every timestep of a held-out
# mesh is test data. These 12 are spread over the whole sequence rather than
# taken consecutively, so the transient does not dominate the statistics.
N_STEPS     = 12
KEEP_FACES  = 30_000        # faces (= 2 opposite edges) kept in memory per mesh
N_SCATTER   = 40_000        # points kept for the density panels
REF_STEPS   = list(range(1, 41))   # window the reference flux is validated on

raw_param_dir = Path("../raw_data/parametric")
flux_cache = processed_data_dir / "parametric_message_flux.npz"


def _r2(X, y):
    """R^2 of the least-squares fit y ~ [X, 1]."""
    X, y = X.double(), y.double()
    X = torch.cat([X, torch.ones(X.shape[0], 1, dtype=X.dtype)], dim=1)
    beta = torch.linalg.lstsq(X, y[:, None]).solution
    ss_res = ((y[:, None] - X @ beta) ** 2).sum()
    ss_tot = ((y - y.mean()) ** 2).sum()
    return float(1.0 - ss_res / ss_tot.clamp(min=1e-30))


def load_mesh(name):
    """(static_graph, T_sequence) for one parametric case, cached to disk.

    Parsing an OpenFOAM case is by far the slowest step here, and the raw
    geometry is the same for both checkpoints, so it is parsed once per mesh.
    """
    cache = (processed_data_dir / "parametric_preproc_cache"
             / f"{name}__T__{'-'.join(EXCLUDED)}.pt")
    if cache.exists():
        blob = torch.load(cache, weights_only=False)
        return blob["graph"], blob["T"]
    g = build_static_graph(str(raw_param_dir / name), EXCLUDED)
    T = load_fields(str(raw_param_dir / name), "T", excluded_patches=EXCLUDED)
    cache.parent.mkdir(exist_ok=True)
    torch.save({"graph": g, "T": T}, cache)
    return g, T


def compute_flux_stats():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    out = {}
    for mesh in FLUX_MESHES:
        graph_full, T_seq = load_mesh(mesh)
        case_dir = str(raw_param_dir / mesh)
        times = filter_of_time_directories(case_dir)

        ei, ea = graph_full.edge_index, graph_full.edge_attr   # RAW geometry
        N, E = graph_full.num_nodes, ei.shape[1]
        n_int = int((graph_full.node_attr[:, 0] == 1.0).sum())
        rev = paired_edge_index(ei)
        internal = (ei[0] < n_int) & (ei[1] < n_int)
        cos = ea[:, 9].double()

        # -- how trustworthy is the reference? Fit V_i per cell in
        #    V_i dT_i/dt = sum_j F_ij and count the impossible (V_i < 0) fits,
        #    for the orthogonal flux and for the non-orth-corrected one.
        dt = float(times[2]) - float(times[1])
        gradT_ref = load_cell_gradient(case_dir, [times[i] for i in REF_STEPS])
        dTdt = (T_seq[[i + 1 for i in REF_STEPS]].double()
                - T_seq[[i - 1 for i in REF_STEPS]].double()) / (2 * dt)
        worst = torch.full((n_int,), 2.0, dtype=torch.double)   # worst face cos per cell
        into_cell = ei[1] < n_int
        worst.scatter_reduce_(0, ei[1][into_cell], cos[into_cell], reduce="amin")
        for tag, F_ref in (
                ("orth", fv_flux(ei, ea, T_seq[REF_STEPS].double(), DT=DT_PARAM)),
                ("corr", fv_flux_corrected(ei, ea, T_seq[REF_STEPS].double(),
                                           gradT_ref, DT=DT_PARAM,
                                           n_internal_nodes=n_int))):
            Fn = torch.zeros(len(REF_STEPS), N, dtype=torch.double)
            Fn.index_add_(1, ei[1], F_ref)
            Fn, d = Fn[:, :n_int], dTdt[:, :n_int]
            V = (d * Fn).sum(0) / (d * d).sum(0).clamp(min=1e-300)
            out[f"ref__{mesh}__{tag}_r2"] = float(
                1 - ((Fn - V[None] * d) ** 2).sum() / (Fn - Fn.mean(0)).pow(2).sum())
            out[f"ref__{mesh}__{tag}_negV"] = float((V < 0).double().mean())
            out[f"ref__{mesh}__{tag}_bins"] = np.array(
                [[int(sel.sum()), float((V[sel] < 0).double().mean())]
                 for sel in ((worst > lo) & (worst <= hi)
                             for lo, hi in ((0.999, 2.0), (0.99, 0.999),
                                            (0.95, 0.99), (-1.0, 0.95)))])

        # -- edges held in memory: whole faces, both halves internal, so that
        #    the antisymmetry test below has both m_ij and m_ji.
        canon = torch.nonzero((rev > torch.arange(E)) & internal
                              & internal[rev]).squeeze(1)
        n_faces = min(KEEP_FACES, canon.numel())
        sel = canon[torch.randperm(canon.numel(),
                                   generator=torch.Generator().manual_seed(0))[:n_faces]]
        kept = torch.cat([sel, rev[sel]])
        rev_local = torch.cat([torch.arange(n_faces) + n_faces,
                               torch.arange(n_faces)])
        near = cos[kept] > 0.99
        out[f"{mesh}__n_edges"] = E
        out[f"{mesh}__n_internal"] = int(internal.sum())
        out[f"{mesh}__n_kept"] = int(kept.numel())
        out[f"{mesh}__n_kept_near"] = int(near.sum())

        # -- the reference flux over the probed timesteps
        steps = np.linspace(1, len(T_seq) - 2, N_STEPS).round().astype(int).tolist()
        gradT = load_cell_gradient(case_dir, [times[i] for i in steps])
        T_at = T_seq[steps].double()
        F = fv_flux_corrected(ei, ea, T_at, gradT, DT=DT_PARAM,
                              n_internal_nodes=n_int)            # (S, E)
        F_node = torch.zeros(N_STEPS, N, dtype=torch.double)
        F_node.index_add_(1, ei[1], F)                           # (S, N)
        # F is w*dT plus the correction, so regressing on each factor separately
        # says WHICH one the message picked up.
        tgts = {"F": F[:, kept].reshape(-1),
                "dT": (T_at[:, ei[0]] - T_at[:, ei[1]])[:, kept].reshape(-1),
                "w": (DT_PARAM * ea[:, 7].double()
                      / (ea[:, 3].double() * cos.clamp(min=0.05)))[kept].repeat(N_STEPS)}

        for label, exp in FLUX_EXPS.items():
            graph = graph_full.clone()
            if exp.endswith("_nofv"):
                graph.edge_attr = graph.edge_attr[:, :4]
            # The normalizer was fitted on the TRAINING meshes and saved with the
            # checkpoint; refitting it here would leak the held-out geometry.
            norm = FeatureNormalizer()
            norm.load(processed_data_dir / exp / "checkpoints" / "normalizer.pt")
            ds = SingleMeshDataset(T_seq, graph, norm, HISTORY)

            model = FVSurrogate(
                in_node_feat=HISTORY + graph.node_attr.shape[1],
                in_edge_feat=graph.edge_attr.shape[1],
                hidden_dim=64, out_dim=1, n_mp_layers=1,
            ).to(device)
            model.load_state_dict(torch.load(
                processed_data_dir / exp / "checkpoints" / "model.pt",
                map_location=device))
            model.eval()

            # history=1, so sample i is built from T[i] — the field the flux
            # must be evaluated on. Reduce every step immediately: the full
            # (E, 128) message is dropped as soon as `kept` is sliced out of it.
            msg_kept, aggr_all = [], []
            probe = MessageProbe(model)
            with probe, torch.no_grad():
                for i in steps:
                    model(ds[i].to(device))
                    msg_kept.append(probe.messages[0][kept])
                    aggr_all.append(probe.aggregated[0])
                    probe.clear()
            msg = torch.stack(msg_kept)                 # (S, kept, C)
            aggr = torch.stack(aggr_all)                # (S, N, C)

            # antisymmetric energy fraction of m_ij across the two halves of a face
            S_, A_ = 0.5 * (msg + msg[:, rev_local]), 0.5 * (msg - msg[:, rev_local])
            sa = A_.pow(2).sum(dim=(0, 1), dtype=torch.float64)
            ss = S_.pow(2).sum(dim=(0, 1), dtype=torch.float64)
            out[f"{exp}__{mesh}__anti"] = (sa / (sa + ss)).numpy()      # (C,)

            # linear read-out of each target from the full 128-channel message
            m_flat = msg.reshape(-1, msg.shape[-1]).double()
            out[f"{exp}__{mesh}__r2"] = np.array(
                [_r2(m_flat, t) for t in tgts.values()])
            # control: the near-orthogonal edges, where even the uncorrected
            # flux is exact and the reference cannot be blamed.
            nf = near.repeat(N_STEPS)
            out[f"{exp}__{mesh}__r2_near"] = np.array(
                [_r2(m_flat[nf], t[nf]) for t in tgts.values()])
            # node level: sum_j m_ij vs the net flux that drives dT_i/dt
            out[f"{exp}__{mesh}__r2_node"] = _r2(
                aggr[:, :n_int].reshape(-1, msg.shape[-1]).double(),
                F_node[:, :n_int].reshape(-1))

            # best-correlated channel, and a subsample for the density panels
            mc = m_flat - m_flat.mean(0, keepdim=True)
            Fc = tgts["F"] - tgts["F"].mean()
            r = (mc * Fc[:, None]).sum(0) / (mc.norm(dim=0) * Fc.norm()).clamp(min=1e-30)
            best = int(r.abs().argmax())
            pick = torch.randperm(m_flat.shape[0],
                                  generator=torch.Generator().manual_seed(0))[:N_SCATTER]
            out[f"{exp}__{mesh}__best_ch"] = best
            out[f"{exp}__{mesh}__best_r"] = float(r[best])
            out[f"{exp}__{mesh}__sc_m"] = m_flat[pick, best].float().numpy()
            out[f"{exp}__{mesh}__sc_F"] = tgts["F"][pick].float().numpy()
            out[f"{exp}__{mesh}__sc_dT"] = tgts["dT"][pick].float().numpy()
            del msg, aggr, m_flat, probe, model
    return out


if flux_cache.exists():
    flux = dict(np.load(flux_cache, allow_pickle=False))
else:
    flux = compute_flux_stats()
    np.savez_compressed(flux_cache, **flux)

print(f"{'mesh':<11}{'model':<7}{'ch':>4}{'r':>8}{'R2 F':>7}{'R2 dT':>7}{'R2 w':>7}"
      f"{'node':>7}{'anti':>7}   R2 F/dT/w on cos>0.99")
for mesh in FLUX_MESHES:
    print(f"{mesh}  reference: V_i fit R^2 "
          f"{float(flux[f'ref__{mesh}__orth_r2']):.3f} (orthogonal) -> "
          f"{float(flux[f'ref__{mesh}__corr_r2']):.3f} (corrected), V_i < 0 on "
          f"{100*float(flux[f'ref__{mesh}__orth_negV']):.0f}% -> "
          f"{100*float(flux[f'ref__{mesh}__corr_negV']):.0f}% of cells")
    for label, exp in FLUX_EXPS.items():
        print(f"{mesh:<11}{label:<7}{int(flux[f'{exp}__{mesh}__best_ch']):>4}"
              f"{float(flux[f'{exp}__{mesh}__best_r']):>+8.3f}"
              + "".join(f"{v:>7.3f}" for v in flux[f"{exp}__{mesh}__r2"])
              + f"{float(flux[f'{exp}__{mesh}__r2_node']):>7.3f}"
              f"{flux[f'{exp}__{mesh}__anti'].mean():>7.3f}   "
              + "/".join(f"{v:.2f}" for v in flux[f"{exp}__{mesh}__r2_near"]))

### Is the reference trustworthy?

Checked against the actual `laplacianFoam` solution before reading anything into the
comparison. The geometry is exact: every edge has its reverse partner,
$S_{f,ij} = -S_{f,ji}$ to machine zero, and $\cos(S_f, d) > 0$ everywhere — internal
minimum 0.63 on `model_003` and 0.74 on `model_011`, down to 0.11 on the boundary edges,
whose column `build_static_graph` recomputes instead of leaving it at OpenFOAM's
hardcoded 1.0.

**The orthogonal flux alone is not good enough on these meshes.** Fitting $V_i$ per cell
in $V_i\,\mathrm{d}T_i/\mathrm{d}t = \sum_j F_{ij}$ over $t = 0.025\ldots1.0$ and counting
the physically impossible negative fits, pooled over the four held-out meshes:

| worst face $\cos$ of cell | cells | $V_i < 0$, orthogonal | $V_i < 0$, corrected |
|---|---|---|---|
| $> 0.999$ | 5803 | 4.2 % | 4.1 % |
| $0.99 - 0.999$ | 22544 | 5.4 % | 2.7 % |
| $0.95 - 0.99$ | 29157 | 21.3 % | 1.7 % |
| $< 0.95$ | 52546 | 25.7 % | 2.3 % |

These are `snappyHexMesh` geometries: only about a quarter of the cells have all their
faces above $\cos = 0.99$, so the flange's fix — restricting to near-orthogonal edges —
would throw most of the mesh away here. Instead the explicit non-orthogonal correction
$k_f\cdot(\nabla T)_f$ is added back with OpenFOAM's **own** cell gradient, which
`laplacianFoam` writes at every write time (`gradTx/y/z`), via
`message_probe.fv_flux_corrected`. That closes the balance to $R^2 = 0.995 - 0.996$ per
mesh, against $0.79 - 0.84$ uncorrected, and drops the impossible fits from ~20 % of
cells to 0.1 - 4.7 %.

So everything below runs on **all internal edges**, not just the orthogonal ones. Two
approximations survive in the correction — the face gradient is a 0.5/0.5 midpoint rather
than OpenFOAM's `linear` weights, and the correction is applied on internal faces only —
and they are what the residual few percent of bad cells are. The near-orthogonal control
($\cos > 0.99$, about 43k of the 60k edges kept per mesh) is computed alongside and
raises every $R^2$ below by 0.01 - 0.10. It changes no conclusion — the only FV vs no-FV
comparison it flips is the near-tied $\Delta T$ on `model_007` — so nothing here rests on
the correction being perfect.

In [13]:
AX_COLOR = "#1e2a2c"
LEGEND_BG = "#f0eee9"
PLOT1_COLOR = "#5e3d80"
PLOT2_COLOR = "#174535"

px = 1/plt.rcParams['figure.dpi']  # pixel in inches

fig, axs = plt.subplots(2, 2, figsize=(1400*px, 900*px), constrained_layout=True)

# Transparent background
fig.patch.set_alpha(0)

bins = np.linspace(0, 1, 31)
for ax, mesh in zip(axs.ravel(), FLUX_MESHES):
    ax.patch.set_alpha(0)
    ax.title.set_color(AX_COLOR)
    ax.xaxis.label.set_color(AX_COLOR)
    ax.yaxis.label.set_color(AX_COLOR)
    ax.tick_params(colors=AX_COLOR, labelsize=14)
    for spine in ax.spines.values():
        spine.set_edgecolor(AX_COLOR)

    for (label, exp), color, ls in zip(FLUX_EXPS.items(),
                                       (PLOT1_COLOR, PLOT2_COLOR), ("-", "--")):
        ax.hist(flux[f"{exp}__{mesh}__anti"], bins=bins, histtype="step",
                linewidth=2.5, color=color, linestyle=ls, label=label)

    # The FV flux sits exactly at 1.0 — that is what makes the scheme conservative.
    ax.axvline(1.0, color=AX_COLOR, linewidth=2.5)
    ax.annotate("FV flux = 1.0\n(conservative)", xy=(1.0, 0.97),
                xycoords=("data", "axes fraction"), xytext=(-10, 0),
                textcoords="offset points", ha="right", va="top",
                fontsize=13, color=AX_COLOR)

    legend = ax.legend(fontsize=15, loc="upper center")
    legend.get_frame().set_facecolor(LEGEND_BG)
    legend.get_frame().set_edgecolor(AX_COLOR)
    legend.get_frame().set_alpha(1)
    for text in legend.get_texts():
        text.set_color(AX_COLOR)

    ax.set_xlim(0, 1.05)
    ax.set_title(mesh, fontsize=19)
    ax.set_xlabel(r"Antisymmetric energy fraction  $\|A\|^2/(\|S\|^2+\|A\|^2)$",
                  fontsize=15)
    ax.set_ylabel("Message channels", fontsize=15)
    ax.grid(True, which="both", ls="-", linewidth=0.5, color=AX_COLOR, alpha=0.25)

fig.suptitle("Is the message antisymmetric across a face?",
             fontsize=24, color=AX_COLOR)

# plt.savefig("../outputs/plots/parametric_message_antisymmetry.png", dpi=300, transparent=True)

In [14]:
AX_COLOR = "#1e2a2c"
LEGEND_BG = "#f0eee9"
PLOT1_COLOR = "#5e3d80"
PLOT2_COLOR = "#174535"

px = 1/plt.rcParams['figure.dpi']  # pixel in inches

fig, axs = plt.subplots(2, 2, figsize=(1400*px, 900*px), constrained_layout=True)
fig.patch.set_alpha(0)

# F is w*dT plus the non-orthogonal correction, so regressing on each factor
# separately says WHICH one the message picked up.
targets = [r"$F$", r"$\Delta T = T_j - T_i$", r"$w = D_T|S_f|/(|d|\cos)$"]
x = np.arange(len(targets))
width = 0.28

for ax, mesh in zip(axs.ravel(), FLUX_MESHES):
    ax.patch.set_alpha(0)
    ax.title.set_color(AX_COLOR)
    ax.xaxis.label.set_color(AX_COLOR)
    ax.yaxis.label.set_color(AX_COLOR)
    ax.tick_params(colors=AX_COLOR, labelsize=14)
    for spine in ax.spines.values():
        spine.set_edgecolor(AX_COLOR)

    for i, ((label, exp), color) in enumerate(zip(FLUX_EXPS.items(),
                                                  (PLOT1_COLOR, PLOT2_COLOR))):
        vals = flux[f"{exp}__{mesh}__r2"]
        off = (i - 0.5) * (width + 0.02)      # 2% surface gap between the pair
        bars = ax.bar(x + off, vals, width, color=color, label=label)
        for b, v in zip(bars, vals):
            ax.annotate(f"{v:.2f}", (b.get_x() + b.get_width()/2, v),
                        xytext=(0, 4), textcoords="offset points",
                        ha="center", fontsize=13, color=AX_COLOR)

    legend = ax.legend(fontsize=15)
    legend.get_frame().set_facecolor(LEGEND_BG)
    legend.get_frame().set_edgecolor(AX_COLOR)
    legend.get_frame().set_alpha(1)
    for text in legend.get_texts():
        text.set_color(AX_COLOR)

    ax.set_xticks(x)
    ax.set_xticklabels(targets, fontsize=15)
    ax.set_ylim(0, 0.82)
    ax.set_title(f"{mesh}  —  {int(flux[f'{mesh}__n_kept'])//1000}k of "
                 f"{int(flux[f'{mesh}__n_internal'])//1000}k internal edges",
                 fontsize=17)
    ax.set_ylabel(r"$R^2$  (all 128 channels $\rightarrow$ target)", fontsize=15)
    ax.grid(True, axis="y", ls="-", linewidth=0.5, color=AX_COLOR, alpha=0.25)
    ax.set_axisbelow(True)

fig.suptitle(f"What the message linearly encodes  ({N_STEPS} probed timesteps)",
             fontsize=24, color=AX_COLOR)

# plt.savefig("../outputs/plots/parametric_message_r2.png", dpi=300, transparent=True)

In [15]:
from matplotlib.colors import LinearSegmentedColormap

AX_COLOR = "#1e2a2c"
LEGEND_BG = "#f0eee9"
PLOT1_COLOR = "#5e3d80"

px = 1/plt.rcParams['figure.dpi']  # pixel in inches

# Sequential ramp for point density: one hue, light -> dark.
DENSITY_CMAP = LinearSegmentedColormap.from_list(
    "fvgn_density", ["#f0eee9", "#b9a8cb", PLOT1_COLOR, "#2b1a3c"])

# Columns: held-out meshes. Rows: the flux, and the bare temperature difference
# that is one of its two factors.
fig, axs = plt.subplots(2, len(FLUX_MESHES),
                        figsize=(1600*px, 800*px), constrained_layout=True)
fig.patch.set_alpha(0)

exp = FLUX_EXPS["FV"]
for col, mesh in enumerate(FLUX_MESHES):
    ch = int(flux[f"{exp}__{mesh}__best_ch"])
    m = flux[f"{exp}__{mesh}__sc_m"]
    panels = [("$F$  [K m$^3$ s$^{-1}$]", flux[f"{exp}__{mesh}__sc_F"]),
              (r"$\Delta T$  [K]", flux[f"{exp}__{mesh}__sc_dT"])]

    for row, (xlabel, xv) in enumerate(panels):
        ax = axs[row, col]
        ax.patch.set_alpha(0)
        ax.title.set_color(AX_COLOR)
        ax.xaxis.label.set_color(AX_COLOR)
        ax.yaxis.label.set_color(AX_COLOR)
        ax.tick_params(colors=AX_COLOR, labelsize=13)
        for spine in ax.spines.values():
            spine.set_edgecolor(AX_COLOR)

        hb = ax.hexbin(xv, m, gridsize=50, bins="log", cmap=DENSITY_CMAP,
                       mincnt=1, linewidths=0)
        r = np.corrcoef(xv, m)[0, 1]
        if row == 0:
            ax.set_title(f"{mesh} · channel {ch}", fontsize=17)
        ax.annotate(f"$r = {r:+.2f}$", xy=(0.03, 0.95), xycoords="axes fraction",
                    va="top", fontsize=15, color=AX_COLOR)
        ax.set_xlabel(xlabel, fontsize=15)
        ax.ticklabel_format(axis="x", style="sci", scilimits=(-2, 3))
        ax.grid(True, ls="-", linewidth=0.5, color=AX_COLOR, alpha=0.25)
        ax.set_axisbelow(True)
        if col == 0:
            ax.set_ylabel("message value", fontsize=15)

cb = fig.colorbar(hb, ax=axs, pad=0.02)
cb.set_label("edge-samples per bin", fontsize=14, color=AX_COLOR)
cb.ax.tick_params(colors=AX_COLOR, labelsize=12)
cb.outline.set_edgecolor(AX_COLOR)

fig.suptitle("Most flux-correlated channel of the FV model",
             fontsize=24, color=AX_COLOR)

# plt.savefig("../outputs/plots/parametric_message_density.png", dpi=300, transparent=True)

### What comes out

**The message is not a conservative flux.** A true FV flux is exactly antisymmetric
across a face, $F_{ij} = -F_{ji}$ — that is what makes the scheme conservative — so its
antisymmetric energy fraction is 1.0. The learned message averages **0.17 - 0.28** across
channels on all four held-out meshes: roughly three quarters of its energy is
*symmetric*, the opposite of a flux. The most antisymmetric single channel reaches only
0.50 - 0.62 (FV) and 0.76 - 0.81 (no-FV). The no-FV message is consistently the more
antisymmetric of the two — 9 to 20 channels above 0.5, against 0 to 5 for the FV model —
which buys it nothing anywhere else below.

**What linear structure it carries tracks $\Delta T$ more than $F$.** Regressing each
target on all 128 channels, the FV model gives $R^2 = 0.34 - 0.42$ for the flux against
$0.48 - 0.52$ for the bare temperature difference. The network leans on the driving
difference more than on the FV weighting that turns it into a flux, though it carries a
substantial amount of both.

**Here the FV features do buy something.** The geometric weight
$w = D_T|S_f|/(|d|\cos)$, which only the FV model's edge features encode, reads out at
$R^2 = 0.45 - 0.64$ from the FV message against $0.08 - 0.20$ from the no-FV one — and
that lead carries through to the flux itself on **every** mesh (0.34 - 0.42 vs
0.29 - 0.32) and to the node level (0.59 - 0.69 vs 0.50 - 0.56, a gain of 0.06 - 0.15).
The no-FV model compensates where it can: with no geometry to lean on, it matches or
beats the FV model on the bare $\Delta T$ on `model_003` and, narrowly, `model_007` — the
one thing left for it to encode. This is the opposite of what the same test gives on the
flange (cached in `flange_message_flux.npz`), where the FV model led only on $w$ and lost
on both $F$ and $\Delta T$.

**The aggregate is far more flux-like than any single message.** At node level, where it
actually matters, $\sum_j m_{ij}$ explains $R^2 = 0.59 - 0.69$ of the net flux
$\sum_j F_{ij}$ that drives $\mathrm{d}T_i/\mathrm{d}t$ — 1.5 to 1.9 times the per-edge
number. The network gets the cell balance broadly right while splitting it over the faces
in a way that is not, face by face, a flux. All of this holds on `model_010` and
`model_011`, whose 2-hole / 3-hole geometries postdate both checkpoints, so it is not an
artefact of the recorded test split.

The density panels show what the correlations leave out: a dense vertical spine at
$F \approx 0$, where the channel spans most of its range while the flux does not move at
all, and a cloud that widens with $|\Delta T|$ rather than following a line.

### Two things this does *not* show

- These are **linear** read-outs of a 128-dimensional message that a **nonlinear** MLP
  consumes. A low $R^2$ bounds how *directly* flux-like the message is; it does not prove
  the flux is unrecoverable from it.
- Both checkpoints are `n_mp_layers=1`, so a single message has to do all the work. A
  deeper stack has no reason to put the flux in layer 0.